# Step 1 – Base heating network (DHNx)
**MJ2505 Project 2 – District heating network optimisation**

| | |
|---|---|
| **Goal** | Supply **5 consumers** from **2 heat producers** using **2 pipe types (DN-30, DN-50)**. DHNx decides which candidate pipes are built and which DN each one gets. |
| **Location** | Real street grid around Odenplan, Vasastan, Stockholm (coordinates from Google Maps) |
| **Reads** | `data/Demand profiles_heating.xlsx` · `data/pipe_types.csv` (Didrik) |
| **Writes** | `data/step1/` (DHNx input CSVs, scaled demand) · `results/step1/` (pipe results, summary, maps) |
| **Grading** | E – build/run the base network, verify both pipe types are used, write down the design problem, justify all inputs |
| **Tutorial reference** | `DHNx Tutorial/DHNx.ipynb` → *introduction* and *discrete_DN_numbers* |

**How to use:** run all cells from top to bottom. The very first time on a new computer, section 0 creates the Python environment.

DN means *diameter nominal*, the nominal inner pipe diameter in mm.

## 0. Setup (run once per computer)

### 0.1 Environment setup cell
**What this cell does**
1. Checks whether all required packages are importable in the *current* kernel → if yes, prints ✅ and does nothing else.
2. If not, it creates a virtual environment `.venv/` inside this repo, installs `requirements.txt` into it and registers it as a Jupyter kernel called **"Python (MJ2505 DH project)"**.
3. It then stops and asks you to **switch kernel** (VS Code: *Select Kernel → Python Environments → .venv*; JupyterLab: *Kernel → Change kernel*) and run the notebook again. From then on the cell just prints ✅.

**Solver:** the course uses **CBC** (`conda install -c conda-forge coin-or-cbc`). If CBC is installed it is used automatically; otherwise the pip-installable **HiGHS** solver is used, so the notebooks run without conda. Both give the same optimum.

Needs Python **3.11–3.14** (the course uses 3.12). If the kernel you opened runs another version, the cell looks for an installed Python 3.11–3.14 to build `.venv` with.

In [ ]:
# 0.1 ENVIRONMENT SETUP - safe to re-run, does nothing if everything is already installed
import sys, subprocess, importlib.util, platform, shutil
from pathlib import Path

REPO = Path.cwd()                      # Jupyter starts in the notebook's folder = repo root
VENV = REPO / ".venv"
KERNEL_NAME, KERNEL_DISPLAY = "mj2505-dh", "Python (MJ2505 DH project)"
REQUIRED = ["dhnx", "oemof.solph", "pyomo", "highspy", "pandas", "numpy", "matplotlib", "openpyxl", "ipykernel"]
SUPPORTED = [(3, 12), (3, 13), (3, 11), (3, 14)]      # preference order


def is_installed(pkg):
    try:
        return importlib.util.find_spec(pkg) is not None
    except ModuleNotFoundError:
        return False


def python_version(cmd):
    try:
        out = subprocess.run(cmd + ["-c", "import sys; print(sys.version_info[0], sys.version_info[1])"],
                             capture_output=True, text=True, timeout=20)
        return tuple(int(x) for x in out.stdout.split()) if out.returncode == 0 else None
    except (OSError, subprocess.TimeoutExpired, ValueError):
        return None


def find_base_python():
    # this kernel's Python if supported, otherwise another installed Python 3.11-3.14
    if sys.version_info[:2] in SUPPORTED:
        return [sys.executable]
    candidates = [["py", f"-{a}.{b}"] for a, b in SUPPORTED] if platform.system() == "Windows" else \
                 [[exe] for a, b in SUPPORTED if (exe := shutil.which(f"python{a}.{b}"))]
    for cmd in candidates:
        if python_version(cmd) in SUPPORTED:
            return cmd
    raise RuntimeError("No Python 3.11-3.14 found. Install Python 3.12 from python.org and re-run this cell.")


missing = [p for p in REQUIRED if not is_installed(p)]
venv_python = VENV / ("Scripts/python.exe" if platform.system() == "Windows" else "bin/python")
running_in_project_venv = Path(sys.prefix).resolve() == VENV.resolve()

if not missing:
    print(f"✅ Environment OK - Python {sys.version.split()[0]} ({sys.executable})")

elif running_in_project_venv:
    print("Installing missing packages into the project venv:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
    print("✅ Done - continue running the notebook.")

else:
    if not venv_python.exists():
        base = find_base_python()
        print("Creating virtual environment", VENV, "with", " ".join(base))
        subprocess.check_call(base + ["-m", "venv", str(VENV)])
    print("Installing requirements.txt into .venv (a few minutes the first time) ...")
    subprocess.check_call([str(venv_python), "-m", "pip", "install", "-q", "--upgrade", "pip"])
    subprocess.check_call([str(venv_python), "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
    subprocess.check_call([str(venv_python), "-m", "ipykernel", "install", "--user",
                           "--name", KERNEL_NAME, "--display-name", KERNEL_DISPLAY])
    raise SystemExit(f"\n✅ Environment created.\n👉 Now switch the notebook kernel to '{KERNEL_DISPLAY}' "
                     "(or the '.venv' interpreter), restart it and run the notebook again from the top.")

### 0.2 Imports and diagnostic
All project code is in `helpers.py`; section 4 holds the Vasastan network and the DHNx runs. The messages *"Need to install geopandas/shapely/osmnx/CoolProp ..."* printed by DHNx are harmless.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

import helpers as h      # section 0: assumptions, 4: Vasastan network + DHNx runs
from helpers import DATA, RESULTS, SOLVER

pd.set_option("display.precision", 2)
pd.set_option("display.width", 160)
OUT = RESULTS / "step1"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Diagnostic (from the course "Prerequisites" document): which Python / packages / solver are used?
import sys, shutil
import pandas as pd
import oemof.solph as solph
import dhnx
from helpers import SOLVER

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("DHNx version:     ", dhnx.__version__)
print("oemof.solph:      ", solph.__version__)
print("pandas:           ", pd.__version__)
print("Solver used:      ", SOLVER, "->", shutil.which("cbc") or "HiGHS via the 'highspy' pip package")

## 1. Assumptions

| Assumption | Value | Why |
|---|---|---|
| Country | Sweden | Needed for prices and emission factors in Step 3 |
| Demand units | kW (hourly mean power) | The Excel file gives no units; kW gives realistic annual totals |
| Demand scale factor | 0.4 on every profile, in every step | See below |
| Design case | Every consumer at its own annual peak at the same time (DHNx `simultaneity` = 1) | Standard, conservative network design |
| Pipe types | Didrik's table (`data/pipe_types.csv`) | Capacities match ΔT = 40 K (80/40 °C, `T_SUPPLY`/`T_RETURN` in `helpers.py`) and about 100 Pa/m pressure gradient, checked below |
| Pipe cost | Overnight investment in €/m, not annualised | Producers are free in DHNx, so annualising with `epc()` would multiply every pipe cost by the same factor: same design, only the objective value changes |
| Producers | Unlimited heat at zero cost inside DHNx | DHNx only designs the network; production costs come in Step 3 |

**Why the demand is scaled by 0.4.** The assignment fixes the pipe range: DN-30 and DN-50 here, at most DN-60 in Step 2. With Didrik's physically based capacities, one DN-50 carries at most 208.9 kW. Each producer is connected through one pipe, so two producers can deliver at most 2 × 208.9 = 418 kW. The five unscaled profiles peak at 873 kW together, which makes the problem infeasible (tested). The profiles are therefore used as **shapes of building types**, scaled by one common factor. With 0.4, the design load is 349 kW, which leaves a 16 % margin. Step 3 must use the same scaled demand (`data/step2/demand_selected.csv`) so that all steps stay consistent.

ΔT is the temperature difference between supply and return water. Pa/m is the pressure drop per metre of pipe, which designers keep low to limit pumping energy.

In [ ]:
print("Country:", h.COUNTRY, "| demand scale:", h.DEMAND_SCALE, "| simultaneity:", h.SIMULTANEITY)

pipe_types = pd.read_csv(h.PIPE_TYPES_CSV)
pipe_types = pipe_types[pipe_types.label_3.isin(h.PIPE_TYPES_BY_STEP[1])].copy()

# Physics check of Didrik's capacities: P = rho * v * A * cp * dT, with the flow velocity v
# set by a pressure gradient R = f * rho * v^2 / (2 D)  ->  v = sqrt(2 D R / (f rho))
D = pipe_types.label_3.str[3:].astype(float) / 1000          # m (nominal = inner diameter)
R, f, rho, cp, dT = 100, 0.025, 980, 4.18, 40                 # Pa/m, friction factor, kg/m3, kJ/kgK, K
v = np.sqrt(2 * D * R / (f * rho))
pipe_types["velocity_m_s"] = v
pipe_types["physics_check_kW"] = rho * v * np.pi * D**2 / 4 * cp * dT
pipe_types["loss_W_per_m"] = pipe_types.l_factor_fix * 1000
pipe_table = pipe_types.rename(columns={"label_3": "type", "cap_max": "capacity_kW",
                                         "fix_costs": "cost_EUR_per_m"})[
    ["type", "capacity_kW", "physics_check_kW", "velocity_m_s", "cost_EUR_per_m", "loss_W_per_m"]]
pipe_table

The capacities in Didrik's table match the physics check within a few percent, so the table can be defended as "ΔT = 40 K and a design pressure gradient of about 100 Pa/m".

In DHNx terms:
- `fix_costs` is the investment cost per metre, paid once a pipe of that type is built (€/m).
- `l_factor_fix` is the heat loss per metre (kW/m).
- `capex_pipes = 0` and `l_factor = 0` mean neither cost nor loss depends on how much heat flows.
- `cap_min` is just below `cap_max`, so each type behaves as one fixed size.

## 2. Consumers, producers and demand

In [ ]:
inp = h.write_step_inputs(1)             # writes data/step1/twn_data, invest_data, demand_selected.csv
demand = inp["demand"]                 # hourly kW after scaling

consumers = inp["consumers"][["key", "name", "building_type", "profile"]].copy()
consumers["peak_kW"] = demand.max().values
consumers["annual_MWh"] = demand.sum().values / 1000
consumers["full_load_hours"] = consumers.annual_MWh * 1000 / consumers.peak_kW
consumers

In [ ]:
inp["producers"][["key", "name", "technology", "note"]]

In [ ]:
sum_peaks = demand.max().sum()
coincident = demand.sum(axis=1).max()
print(f"Sum of individual peaks : {sum_peaks:6.1f} kW  (design load used by DHNx)")
print(f"Coincident peak         : {coincident:6.1f} kW  (highest hour of the summed profile)")
print(f"Coincidence factor      : {coincident / sum_peaks:6.2f}")
print(f"Annual demand           : {demand.sum().sum() / 1000:6.1f} MWh")

fig, ax = plt.subplots(figsize=(11, 3.5))
demand.rolling(24 * 7, center=True).mean().plot(ax=ax, lw=1.5)
ax.set_ylabel("kW (7-day mean)"); ax.set_xlabel("hour of year")
ax.set_title("Step 1 consumer demand profiles (scaled)"); ax.legend(ncol=5, fontsize=8)
h.RESULTS.joinpath("step1").mkdir(parents=True, exist_ok=True)
fig.savefig(h.RESULTS / "step1" / "demand_profiles.png", dpi=150, bbox_inches="tight")

The five profiles have different shapes: winter-peaked residential and school loads, a flat 24/7 hospital load and a hot-water-dominated hotel. Their peaks don't occur in the same hour, so the coincident peak is lower than the sum of peaks. DHNx still designs for the sum (simultaneity = 1). That is conservative, and something to state as a limitation.

## 3. Candidate network

Every street segment drawn below is a *candidate*. DHNx decides which ones to build. The streets form a ladder with loops, so DHNx has real route choices.

In [ ]:
ax = h.plot_dh_network(1, title="Step 1: candidate network (lengths in m)")
plt.gcf().savefig(h.RESULTS / "step1" / "map_candidates.png", dpi=150, bbox_inches="tight")
inp["pipes"][["id", "from_key", "to_key", "street", "length"]].head(19)

## 4. The optimisation problem

DHNx turns the network into a mixed-integer linear program (MILP: a linear optimisation where some variables can only take whole-number values, here 0 or 1). It solves it for **one design hour**.

**Sets**
- $E$: candidate pipes (street segments, producer connections, service pipes). Each can carry heat in either direction $d$.
- $K$: pipe types (DN-30, DN-50).

**Decision variables**
- $y_{e,k,d} \in \{0,1\}$: build pipe type $k$ on edge $e$ in direction $d$.
- $P_{e,k,d}$ [kW]: installed capacity, with $cap_{min,k}\,y \le P \le cap_{max,k}\,y$.
- $q_{e,k,d}$ [kW]: heat flow in the pipe at the design hour, $q \le P$.
- $Q_p$ [kW]: heat fed in by producer $p$.

**Objective:** minimise total pipe investment
$$\min \sum_{e,k,d} L_e \, (c^{fix}_k \, y_{e,k,d} + c^{var}_k \, P_{e,k,d})$$
where $L_e$ is the pipe length in m, $c^{fix}_k$ is `fix_costs` in €/m, and $c^{var}_k$ is `capex_pipes`, which is 0 here. Heat from producers costs nothing in DHNx.

**Constraints**
1. **Heat balance** at every junction, consumer and producer node: inflow = outflow.
2. **Demand:** each consumer receives its design load, `P_heat_max` × simultaneity.
3. **Pipe losses:** heat entering = heat leaving + $L_e \, (l_k \, P + l^{fix}_k \, y)$. With $l_k = 0$, the loss only depends on whether the pipe is built.
4. **Capacity:** flow ≤ installed capacity.
5. **One pipe per edge (added by us):** $\sum_{k,d} y_{e,k,d} \le 1$. DHNx 0.0.4 has no such rule. Without it, the optimiser stacks DN-30 + DN-50 in parallel on one street, and DHNx's own result code then crashes.
6. **One connected network (added by us):** every producer and consumer must be linked through built pipes. Without it, DHNx builds separate islands, because heat is free (shown in section 8). Steps 3–5 assume all producers serve the aggregate demand, which needs one network. This is implemented as a flow of "connectivity units" from producer P0 to every other node, allowed only on built edges.

**Parameters:** pipe lengths (from coordinates), pipe type costs, losses and capacities (Didrik), and the consumer design loads (scaled peaks).

In [ ]:
net = h.run_dhnx(1)                    # connected=True by default
h.solver_info(net)

## 5. Results

In [ ]:
pipes = h.pipe_results(net, 1)
built = pipes[pipes.built]
used_types = set(built.hp_type)
assert used_types == {"DN-30", "DN-50"}, f"Both pipe types must be used, got {used_types}"
print("Both pipe types used:", sorted(used_types))
summary = h.summary(net, 1, pipes)
summary.round(1)

In [ ]:
cols = ["id", "street", "from_key", "to_key", "kind", "length", "hp_type", "capacity",
        "design_flow_kW", "losses", "utilisation", "costs"]
built[cols].rename(columns={"capacity": "capacity_kW", "losses": "loss_kW",
                            "costs": "cost_EUR"}).sort_values(["kind", "hp_type"])

In [ ]:
supply = h.producer_supply(net, 1)
print(f"Demand + losses = {summary['design demand (sum of peaks) [kW]'] + summary['heat losses at design hour [kW]']:.1f} kW,"
      f" producers supply {supply.supply_kW.sum():.1f} kW")
supply

In [ ]:
ax = h.plot_dh_network(1, pipes, title="Step 1: optimised network (numbers = heat flow at design hour, kW)")
plt.gcf().savefig(h.RESULTS / "step1" / "map_result.png", dpi=150, bbox_inches="tight")

## 6. Comparing the two pipe types

In [ ]:
by_type = built.groupby("hp_type").agg(
    pipes=("id", "size"), length_m=("length", "sum"), cost_EUR=("costs", "sum"),
    loss_kW=("losses", "sum"), mean_flow_kW=("design_flow_kW", "mean"),
    mean_utilisation=("utilisation", "mean"))
by_type["used_as"] = built.groupby("hp_type").kind.agg(lambda s: ", ".join(f"{k} x{n}" for k, n in s.value_counts().items()))
pt = pipe_table.set_index("type")
by_type["capacity_kW"] = pt.capacity_kW
by_type["cost_EUR_per_m"] = pt.cost_EUR_per_m
by_type["loss_W_per_m"] = pt.loss_W_per_m
by_type["cost_EUR_per_m_per_kW"] = by_type.cost_EUR_per_m / by_type.capacity_kW
by_type

## 7. Answers to the Step 1 questions

Numbers refer to the run above.

**How does the optimiser decide which pipe type to use where?**
It minimises total investment, length × €/m. Each built pipe must carry the heat flowing through it at the design hour, plus its own loss. DN-30 is the cheaper type (264.1 vs 323.8 €/m, about 18 % less), so the optimiser uses it wherever the flow fits under 53.6 kW. Where the flow is larger, DN-50 is the only option. The flow on each segment is the sum of all demand served "downstream" of it, so the pipe type follows the accumulated load. Heat losses do not enter the objective directly: heat is free in DHNx, so losses only add a little flow the producers must cover.

**How do the two pipe types differ in cost, losses and capacity?**
- **Capacity:** DN-50 carries almost 4 times more (208.9 vs 53.6 kW) for only 23 % more cost per metre.
- **Cost per unit of capacity:** DN-50 is therefore much cheaper per kW (1.55 vs 4.93 €/m per kW).
- **Losses:** almost the same per metre (9.2 vs 8.3 W/m), because losses depend on pipe surface and temperature, not on flow.

So big pipes are cheap per kW, but you pay for their full size even where the flow is small. That is why the optimiser does not simply use DN-50 everywhere.

**Why DN-50 near the start of the network and DN-30 elsewhere?**
Next to a producer, a pipe carries the heat for everything it feeds: about 197 kW out of the heat pump (P0) and 164 kW out of the biomass boiler (P1). Only DN-50 can carry that. Moving outwards, every branch takes heat off the main line, and the flow drops. The end branches carry one building each, the small residential blocks at 20 and 44 kW, so DN-30 suffices. The service pipes to the school, hotel and hospital are DN-50 because their peaks (73–109 kW) exceed DN-30. Step 1 has no in-between size; Step 2 adds DN-40 for exactly these. The link between the two producer areas (F3–F4–F5) is DN-30 and runs almost full (97–100 % utilisation): the optimiser sends only as much heat south as the small pipe allows.

**Binding capacities:** the pipes at or near 100 % utilisation are where the design is tight. Here that is the DN-30 link to Sabbatsberg and the DN-50 trunk out of P0 (94 %).

In [ ]:
binding = built[built.utilisation > 0.9][["street", "from_key", "to_key", "hp_type", "capacity",
                                            "design_flow_kW", "losses", "utilisation"]]
binding

## 8. What DHNx does without the "one connected network" rule

For comparison only, this section shows plain DHNx with only the one-pipe-per-edge rule. Because heat is free, every producer simply feeds its own neighbourhood and no pipe links the areas. The islands are cheaper, but they cannot share heat, so they do not fit the aggregate-demand model of Steps 3–5.

In [ ]:
import networkx as nx
net_free = h.run_dhnx(1, connected=False)
pipes_free = h.pipe_results(net_free, 1)
b_free = pipes_free[pipes_free.built]
islands = nx.number_connected_components(nx.from_pandas_edgelist(b_free, "from_key", "to_key"))
print(f"Without connectivity rule: {islands} islands, investment {b_free.costs.sum():,.0f} EUR")
print(f"With connectivity rule   : 1 network,  investment {built.costs.sum():,.0f} EUR")
print(f"Cost of connecting the two producer areas: {built.costs.sum() - b_free.costs.sum():,.0f} EUR "
      f"({100 * (built.costs.sum() / b_free.costs.sum() - 1):.1f} %)")
ax = h.plot_dh_network(1, pipes_free, title="Step 1 without the connectivity rule (comparison only)")

## 9. Save results

In [ ]:
out = h.RESULTS / "step1"
pipes.to_csv(out / "pipes_result.csv", index=False)
supply.to_csv(out / "producer_supply.csv", index=False)
summary.to_csv(out / "summary.csv", header=["value"])
by_type.to_csv(out / "pipe_types_used.csv")
h.solver_info(net).to_csv(out / "solver_info.csv", header=["value"])
print("Saved to", out.relative_to(h.ROOT))
sorted(p.name for p in out.iterdir())